# Phase 4A — Drift Validation and Temporal-Gap Audit

## Aim

Before estimating or correcting drift, this notebook verifies that the
existing displacement steps correspond to the intended physical frame lags.

The current implementation uses row-based shifting within each trajectory.
If a trajectory contains missing frames, a one-row shift may represent more
than one physical frame.

This notebook will:

1. load and validate the cleaned trajectories;
2. detect duplicate cell-frame observations;
3. measure frame and time gaps inside each trajectory;
4. compare labeled lag values with actual frame differences;
5. determine whether the Phase 3 step implementation requires correction;
6. proceed to drift validation only after the lag definition is verified.

In [1]:
import _path
from _path import PROJECT_ROOT

import numpy as np
import pandas as pd

from src.steps import (
    compute_steps_multi_tau,
    validate_tracks,
)

TRACKS_PATH = PROJECT_ROOT / "data" / "MSC01_tracks_clean.csv"

print("Project root:", PROJECT_ROOT)
print("Input file:", TRACKS_PATH)
print("Input exists:", TRACKS_PATH.exists())

Project root: C:\Users\Abolfazl.PH\Desktop\cell-irreversibility
Input file: C:\Users\Abolfazl.PH\Desktop\cell-irreversibility\data\MSC01_tracks_clean.csv
Input exists: True


In [2]:
tracks = pd.read_csv(TRACKS_PATH)

validate_tracks(tracks)

print("Shape:", tracks.shape)
print("Number of cells:", tracks["cell_id"].nunique())
print("Frame range:", tracks["frame"].min(), "to", tracks["frame"].max())

display(tracks.head())
display(tracks.dtypes)

Shape: (474, 5)
Number of cells: 35
Frame range: 0 to 47


,cell_id,frame,t_min,x_um,y_um
0,0,10,200.0,39.867660,224.393768
1,0,11,220.0,34.776408,227.566565
2,0,12,240.0,31.056212,229.721570
3,0,13,260.0,32.449259,230.256018
4,0,14,280.0,33.368621,230.260754


cell_id      int64
frame        int64
t_min      float64
x_um       float64
y_um       float64
dtype: object

In [3]:
duplicate_mask = tracks.duplicated(
    subset=["cell_id", "frame"],
    keep=False,
)

n_duplicate_rows = int(duplicate_mask.sum())

print("Duplicate cell-frame rows:", n_duplicate_rows)

if n_duplicate_rows > 0:
    display(
        tracks.loc[duplicate_mask]
        .sort_values(["cell_id", "frame"])
    )

Duplicate cell-frame rows: 0


In [4]:
tracks_sorted = (
    tracks
    .sort_values(["cell_id", "frame"])
    .reset_index(drop=True)
    .copy()
)

grouped_tracks = tracks_sorted.groupby(
    "cell_id",
    sort=False,
)

tracks_sorted["frame_gap"] = (
    grouped_tracks["frame"]
    .diff()
    .astype("Int64")
)

tracks_sorted["time_gap_min"] = (
    grouped_tracks["t_min"]
    .diff()
)

gap_counts = (
    tracks_sorted
    .dropna(subset=["frame_gap"])
    .groupby(["frame_gap", "time_gap_min"])
    .size()
    .rename("n_transitions")
    .reset_index()
    .sort_values(["frame_gap", "time_gap_min"])
)

display(gap_counts)

,frame_gap,time_gap_min,n_transitions
0,1,20.0,433
1,2,40.0,6


In [5]:
gap_audit = (
    tracks_sorted
    .dropna(subset=["frame_gap", "time_gap_min"])
    .loc[:, ["cell_id", "frame", "frame_gap", "time_gap_min"]]
    .copy()
)

gap_audit["expected_time_gap_min"] = (
    gap_audit["frame_gap"] * 20.0
)

gap_audit["time_is_consistent"] = np.isclose(
    gap_audit["time_gap_min"],
    gap_audit["expected_time_gap_min"],
)

print(
    "Transitions with inconsistent frame/time calibration:",
    int((~gap_audit["time_is_consistent"]).sum()),
)

display(
    gap_audit.loc[~gap_audit["time_is_consistent"]].head(20)
)

Transitions with inconsistent frame/time calibration: 0


,cell_id,frame,frame_gap,time_gap_min,expected_time_gap_min,time_is_consistent


In [6]:
taus = [1, 2, 4]

steps_current = compute_steps_multi_tau(
    tracks,
    taus=taus,
).copy()

steps_current["actual_frame_gap"] = (
    steps_current["frame_end"]
    - steps_current["frame_start"]
)

steps_current["lag_matches_label"] = (
    steps_current["actual_frame_gap"]
    == steps_current["tau_frames"]
)

lag_audit = (
    steps_current
    .groupby(["tau_frames", "actual_frame_gap"])
    .size()
    .rename("n_steps")
    .reset_index()
    .sort_values(["tau_frames", "actual_frame_gap"])
)

display(lag_audit)

,tau_frames,actual_frame_gap,n_steps
0,1,1,433
1,1,2,6
2,2,2,392
3,2,3,12
4,4,4,329
5,4,5,13
6,4,6,2


In [7]:
lag_summary = (
    steps_current
    .groupby("tau_frames")
    .agg(
        n_steps=("cell_id", "size"),
        n_matching=("lag_matches_label", "sum"),
    )
    .reset_index()
)

lag_summary["n_mismatched"] = (
    lag_summary["n_steps"]
    - lag_summary["n_matching"]
)

lag_summary["percent_mismatched"] = (
    100
    * lag_summary["n_mismatched"]
    / lag_summary["n_steps"]
)

display(lag_summary)

total_mismatched = int(
    (~steps_current["lag_matches_label"]).sum()
)

print("Total mislabeled-lag steps:", total_mismatched)

,tau_frames,n_steps,n_matching,n_mismatched,percent_mismatched
0,1,439,433,6,1.366743
1,2,404,392,12,2.970297
2,4,344,329,15,4.360465


Total mislabeled-lag steps: 33
